# Chapter 5: Embeddings

[Read this chapter online](https://jackluu.io/book/section-1-foundations/ch05-embeddings/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch05-embeddings.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 5: Embeddings

![Where we are in the big picture](../assets/diagrams/ch05-where-we-are.png){ width="756" }
*Figure 5.1: We have numbers, now we give them meaning.*

In the previous chapter, we turned text into tokens (Figure 5.1). Tokens are just arbitrary numbers. To predict what comes next, the model needs to understand how characters relate to each other. In this chapter you will:

- Map simple token IDs to large vectors of numbers.
- Learn how these vectors capture meaning as coordinates.
- Add position information so the model knows the order of characters.

**Words to Know**
    - **Vector**: A list of numbers that acts as a coordinate in a high-dimensional space.
    - **Embedding**: A lookup table that maps a token ID to its vector.

## Theory

### The Problem with IDs

![IDs have no math meaning](../assets/diagrams/ch05-id-problem.png){ width="538" }
*Figure 5.2: Token IDs are just arbitrary labels, not math quantities.*

After tokenization, each character is an integer: `A=13`, `B=14`, `a=39`. But as Figure 5.2 illustrates, feeding these raw integers to a neural network creates a problem. The network learns by multiplying numbers together. It would see `Z` (token 38) as almost three times as big as `A` (token 13). It might learn that `Z` is more important than `A` just because of this accident. But token IDs are arbitrary labels.

Think of it like an employee ID number. Employee 105 is not "five times more employee" than Employee 21. The number alone carries no meaning about their role. We need a way to turn these arbitrary labels into a format the "Numbers Machine" can use to find patterns.

### Vectors as Coordinates of Meaning

![Looking up a token ID to find its embedding vector](../assets/diagrams/ch05-embedding-table.png){ width="498" }
*Figure 5.3: An embedding table maps each ID to a vector of numbers.*

The solution, shown in Figure 5.3, is a lookup table called an **embedding**. It maps each token ID to a vector of floating-point numbers. In our model, we use 128 numbers for each character.

Instead of seeing `39` for `a`, the model sees `[0.55, 0.03, -0.89, ...]`. These 128 numbers act like coordinates. Characters that appear in similar contexts will gradually be moved closer together in this 128-dimensional space during training. The network learns that capital and lowercase versions of a letter behave similarly, so their vectors become similar. This table of profiles is learned from scratch.

### Position Embeddings

![Adding token and position embeddings together](../assets/diagrams/ch05-position-addition.png){ width="408" }
*Figure 5.4: The final representation combines what the token is with where it sits.*

There is one more problem: the model reads all characters at exactly the same time. Without help, it cannot tell the difference between "cat" and "act" because they use the same letters. Order matters.

We fix this with **positional embeddings** (Figure 5.4): a second lookup table indexed by the position of the character in the sequence (0, 1, 2, ...) rather than its token ID. Position 0 gets its own 128-number vector, position 1 gets a different vector, and so on.

We then add the token embedding and the positional embedding together. The network learns to use these combined 128 numbers to represent both "what character is this?" and "where does it appear?". This combined representation completes the "Embeddings" stage in our map (Figure 5.1). The text is now fully converted into rich math vectors, ready for the next stage.

**In Business**
    When you build an assistant that writes in your company's house style, embeddings turn your archive into a landscape of meaning. Token embeddings capture the vocabulary, while the position embeddings give the model the order of the characters, which is what lets it pick up the shape of an invoice or a polite greeting.

## Code

We use `src/ch04_embeddings.py` to create and combine these two tables.

```python
token_emb = nn.Embedding(config.vocab_size, config.n_embd)
    token_embeddings = token_emb(token_ids)
    pos_emb = nn.Embedding(config.block_size, config.n_embd)
    positions = torch.arange(T)
    position_embeddings = pos_emb(positions)
    x = token_embeddings + position_embeddings
```

Here is what happens when we run it:

```python
$ python src/ch04_embeddings.py
--- 2. Looking up embeddings ---
Input token_ids shape : torch.Size([2, 5])
Token embeddings shape: torch.Size([2, 5, 128])
  (B=2, T=5, C=128)
...
--- 3. Positional embeddings ---
Position indices: [0, 1, 2, 3, 4]
Position embeddings shape: torch.Size([5, 128])

--- 4. Combining token + position embeddings ---
Final x shape: torch.Size([2, 5, 128])
  (B=2, T=5, C=128)
```

**What just happened:**

- Line 1 created a lookup table for the 65 characters in our vocabulary.
- Line 2 translated a batch of token IDs into their embedding vectors.
- Line 3 created a second lookup table for the sequence positions.
- Line 6 added the token and position vectors to create the final input `x`.

**Shape Check:** Table 5.1 details the dimensions of our data structures.

**Table 5.1:** Tensor shapes before and after the embedding layer.

| Variable | Shape | Meaning |
|---|---|---|
| `token_ids` | `[B, T]` | Batch size by Time (sequence length). |
| `token_embeddings` | `[B, T, C]` | `C` is the embedding dimension (128). |
| `position_embeddings` | `[T, C]` | One vector per position. |
| `x` | `[B, T, C]` | The combined input for the model. |

## Try It

**Try It**
    Change the batch size `B` or the sequence length `T` in `src/ch04_embeddings.py`. Run the script again. Notice that the output shapes scale automatically, but the parameter counts stay exactly the same. The lookup tables do not care how much text you process at once.

**Watch Out**
    A common mistake is trying to look up a token ID that is larger than the vocabulary size. If your `vocab_size` is 65, the valid IDs are 0 to 64. Passing an ID of 65 will crash the program with an "index out of bounds" error.

## Key Takeaways

- Token IDs are arbitrary labels and cannot be used directly for math.
- Embeddings are learned lookup tables that map token IDs to vectors.
- These vectors act as coordinates that capture relationships and meaning.
- Positional embeddings are added to tell the model the order of the characters.

## Check Your Understanding

1. Why do neural networks struggle with raw token IDs?
2. How many numbers make up a single character's embedding vector in our model?
3. Why do we need positional embeddings in addition to token embeddings?
4. Does the size of the embedding table depend on the batch size?


## Further Reading

**Meaning becomes geometry.** Give every word its own ID number and the model learns nothing from the numbering: "king" sits as far from "queen" as it does from "toaster". This paper trained a deliberately cheap prediction task so that words used in similar company ended up with similar vectors, and did it fast enough to run on billions of words. Embeddings, the subject of Chapter 5, start here, and so does the vector search behind modern recommendation and retrieval.

<div class="refs" markdown>

Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). *Efficient estimation of word representations in vector space* (arXiv:1301.3781). arXiv. https://doi.org/10.48550/arXiv.1301.3781

</div>

---

### `src/ch04_embeddings.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch04_embeddings.py"   # a cell has none, and the file uses it to find the text

"""
Introduce embedding tables and positional embeddings.
This file belongs to Chapter 5.
Run: python src/ch04_embeddings.py
"""
import torch
import torch.nn as nn

import os
import sys

from src.utils.config import GPTConfig

# Settings
config = GPTConfig()

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 5: Embeddings\n")

    print("--- 1. The embedding table ---")
    # Creates a lookup table mapping each vocab ID to a vector
    token_emb = nn.Embedding(config.vocab_size, config.n_embd)

    print(f"Embedding table shape: {token_emb.weight.shape}")
    print(f"  vocab_size = {config.vocab_size}  (one row per character)")
    print(f"  n_embd     = {config.n_embd}  (numbers per character)")

    print("\n--- 2. Looking up embeddings ---")
    B, T = 2, 5
    token_ids = torch.tensor([[3, 14, 7, 2, 50], [10, 22, 45, 1, 8]])

    # Convert token IDs into their corresponding embedding vectors
    token_embeddings = token_emb(token_ids)

    print(f"Input token_ids shape : {token_ids.shape}")
    print(f"Token embeddings shape: {token_embeddings.shape}")
    print(f"  (B={B}, T={T}, C={config.n_embd})")
    print(f"\nFirst token's embedding (first 8 values): "
          f"{token_embeddings[0, 0, :8].tolist()}")

    print("\n--- 3. Positional embeddings ---")
    # Creates a lookup table mapping each position to a vector
    pos_emb = nn.Embedding(config.block_size, config.n_embd)

    # Generate position indices 0, 1, 2, ..., T-1
    positions = torch.arange(T)
    print(f"Position indices: {positions.tolist()}")

    position_embeddings = pos_emb(positions)
    print(f"Position embeddings shape: {position_embeddings.shape}")

    print("\n--- 4. Combining token + position embeddings ---")
    # Add token and position embeddings so the model knows what and where it is
    x = token_embeddings + position_embeddings

    print(f"Final x shape: {x.shape}")
    print(f"  (B={B}, T={T}, C={config.n_embd})")
    print("\nThis tensor x is the input to the transformer blocks.")
    print(
        f"Each of the {B*T} slots has {config.n_embd} numbers describing it."
    )

    print("\n--- Summary ---")
    total_params = (config.vocab_size + config.block_size) * config.n_embd
    print(f"Token embedding parameters  : {config.vocab_size} x {config.n_embd} "
          f"= {config.vocab_size * config.n_embd:,}")
    print(f"Position embedding parameters: {config.block_size} x {config.n_embd} "
          f"= {config.block_size * config.n_embd:,}")
    print(f"Total embedding parameters   : {total_params:,}")
    print("\nEmbeddings done! Ready for Chapter 6.")